# Regression Metrics - R², SSR & MAE Explained

A **detailed, visual, hands-on guide** to the most important regression evaluation metrics:

| Metric | Full Name | What It Tells You |
|--------|-----------|-------------------|
| **SST** | Sum of Squares Total | Total variance in the data |
| **SSR** | Sum of Squares Regression (Explained) | Variance captured by the model |
| **SSE** | Sum of Squares Error (Residual) | Variance the model missed |
| **R²** | Coefficient of Determination | Proportion of variance explained |
| **MAE** | Mean Absolute Error | Average prediction error in original units |

By the end of this notebook you will:
1. Understand the **mathematical intuition** behind each metric
2. Know **when to use which** metric
3. Be able to **compute them from scratch** and verify with scikit-learn
4. See how they behave on **good vs. bad** models

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

np.random.seed(42)
print('All imports ready!')

## 2. Create a Simple Dataset

We'll use a small, easy-to-visualize dataset so every metric calculation is transparent.

In [ ]:
# Simple dataset: Years of Experience vs Salary (in thousands)
X = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]).reshape(-1, 1)
y = np.array([30, 35, 45, 50, 55, 58, 65, 72, 80, 90])  # Salary in $1000s

# Fit a simple linear regression
model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

print(f"Slope (coefficient): {model.coef_[0]:.2f}")
print(f"Intercept: {model.intercept_:.2f}")
print(f"Equation: Salary = {model.coef_[0]:.2f} * Experience + {model.intercept_:.2f}")

# Display data
df = pd.DataFrame({'Experience (yrs)': X.ravel(), 'Actual Salary': y, 'Predicted Salary': np.round(y_pred, 2)})
df['Error (y - ŷ)'] = np.round(y - y_pred, 2)
df

---
## 3. The Big Picture: SST = SSR + SSE

Before diving into individual metrics, understand this fundamental identity:

$$\underbrace{\sum(y_i - \bar{y})^2}_{\text{SST (Total)}} = \underbrace{\sum(\hat{y}_i - \bar{y})^2}_{\text{SSR (Regression/Explained)}} + \underbrace{\sum(y_i - \hat{y}_i)^2}_{\text{SSE (Error/Residual)}}$$

**Think of it as a pie:**
- **SST** = the whole pie (total variation in `y`)
- **SSR** = the slice your model explains
- **SSE** = the slice your model fails to explain (the leftovers / errors)

A perfect model: SSR = SST, SSE = 0  
A terrible model: SSR ≈ 0, SSE ≈ SST

---
## 4. SST — Sum of Squares Total

### What is it?
SST measures the **total variability** in the target variable `y` around its mean.

$$SST = \sum_{i=1}^{n}(y_i - \bar{y})^2$$

### Intuition
If you had **no model at all**, the best guess for any data point would be the mean $\bar{y}$. SST measures how far the actual values are from this "no-model" baseline.

### Key Point
SST has **nothing to do with your model** — it only depends on the data.

In [ ]:
y_mean = np.mean(y)
print(f"Mean of y (ȳ): {y_mean}")

# Calculate SST step by step
deviations_total = y - y_mean
squared_deviations_total = deviations_total ** 2
SST = np.sum(squared_deviations_total)

print(f"\nStep-by-step SST calculation:")
print(f"{'y_i':>6} {'ȳ':>6} {'(y_i - ȳ)':>10} {'(y_i - ȳ)²':>12}")
print("-" * 40)
for i in range(len(y)):
    print(f"{y[i]:6.1f} {y_mean:6.1f} {deviations_total[i]:10.1f} {squared_deviations_total[i]:12.1f}")
print("-" * 40)
print(f"{'SST':>30} = {SST:.2f}")

In [ ]:
# Visualize SST
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X, y, color='blue', s=100, zorder=5, label='Actual values')
ax.axhline(y=y_mean, color='red', linestyle='--', linewidth=2, label=f'Mean (ȳ = {y_mean})')

# Draw vertical lines from each point to the mean
for i in range(len(y)):
    ax.vlines(X[i], y_mean, y[i], color='orange', linewidth=2, alpha=0.7)
    # Draw squares to represent squared deviations
    size = abs(y[i] - y_mean) * 0.08

ax.set_xlabel('Experience (years)')
ax.set_ylabel('Salary ($1000s)')
ax.set_title(f'SST: Total Deviation from Mean\nSST = Σ(yᵢ - ȳ)² = {SST:.1f}', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## 5. SSR — Sum of Squares Regression (Explained Variation)

### What is it?
SSR measures how much of the total variation the **model explains** — i.e., how far the predicted values are from the mean.

$$SSR = \sum_{i=1}^{n}(\hat{y}_i - \bar{y})^2$$

### Intuition
- If the regression line is **flat** at the mean → SSR = 0 (model explains nothing)
- If the regression line passes through every point → SSR = SST (model explains everything)

### Why "Regression" in the name?
Because it measures the variation that is **due to the regression** (the fitted relationship between X and y).

In [ ]:
# Calculate SSR step by step
deviations_regression = y_pred - y_mean
squared_deviations_regression = deviations_regression ** 2
SSR = np.sum(squared_deviations_regression)

print(f"Step-by-step SSR calculation:")
print(f"{'ŷ_i':>8} {'ȳ':>6} {'(ŷ_i - ȳ)':>11} {'(ŷ_i - ȳ)²':>12}")
print("-" * 42)
for i in range(len(y)):
    print(f"{y_pred[i]:8.2f} {y_mean:6.1f} {deviations_regression[i]:11.2f} {squared_deviations_regression[i]:12.2f}")
print("-" * 42)
print(f"{'SSR':>32} = {SSR:.2f}")

In [ ]:
# Visualize SSR
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X, y, color='blue', s=100, zorder=5, label='Actual values', alpha=0.5)
ax.plot(X, y_pred, color='green', linewidth=2, label='Regression line')
ax.axhline(y=y_mean, color='red', linestyle='--', linewidth=2, label=f'Mean (ȳ = {y_mean})')

# Draw vertical lines from mean to predicted values
for i in range(len(y)):
    ax.vlines(X[i], y_mean, y_pred[i], color='green', linewidth=3, alpha=0.6)

ax.set_xlabel('Experience (years)')
ax.set_ylabel('Salary ($1000s)')
ax.set_title(f'SSR: Explained Variation (Model vs Mean)\nSSR = Σ(ŷᵢ - ȳ)² = {SSR:.1f}', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## 6. SSE — Sum of Squares Error (Residual Variation)

### What is it?
SSE measures the variation that the model **fails to explain** — the residuals.

$$SSE = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

### Intuition
- SSE = 0 → model is perfect (every prediction matches the actual value)
- SSE = SST → model is useless (no better than just predicting the mean)

### Key Point
This is exactly what **Ordinary Least Squares (OLS)** minimizes! The "least squares" in OLS refers to minimizing SSE.

In [ ]:
# Calculate SSE step by step
residuals = y - y_pred
squared_residuals = residuals ** 2
SSE = np.sum(squared_residuals)

print(f"Step-by-step SSE calculation:")
print(f"{'y_i':>6} {'ŷ_i':>8} {'(y_i - ŷ_i)':>12} {'(y_i - ŷ_i)²':>14}")
print("-" * 45)
for i in range(len(y)):
    print(f"{y[i]:6.1f} {y_pred[i]:8.2f} {residuals[i]:12.2f} {squared_residuals[i]:14.2f}")
print("-" * 45)
print(f"{'SSE':>33} = {SSE:.2f}")

In [ ]:
# Visualize SSE
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X, y, color='blue', s=100, zorder=5, label='Actual values')
ax.plot(X, y_pred, color='green', linewidth=2, label='Regression line')

# Draw vertical lines from actual to predicted (residuals)
for i in range(len(y)):
    color = 'red' if residuals[i] < 0 else 'purple'
    ax.vlines(X[i], y_pred[i], y[i], color=color, linewidth=2.5, alpha=0.7)

ax.set_xlabel('Experience (years)')
ax.set_ylabel('Salary ($1000s)')
ax.set_title(f'SSE: Residual Variation (Actual vs Predicted)\nSSE = Σ(yᵢ - ŷᵢ)² = {SSE:.1f}', fontsize=14)

purple_patch = mpatches.Patch(color='purple', label='Underprediction (y > ŷ)')
red_patch = mpatches.Patch(color='red', label='Overprediction (y < ŷ)')
ax.legend(handles=ax.get_legend_handles_labels()[0] + [purple_patch, red_patch], fontsize=10)
plt.tight_layout()
plt.show()

---
## 7. Verify: SST = SSR + SSE

In [ ]:
print(f"SST (Total)      = {SST:.4f}")
print(f"SSR (Regression) = {SSR:.4f}")
print(f"SSE (Error)      = {SSE:.4f}")
print(f"SSR + SSE        = {SSR + SSE:.4f}")
print(f"\nSST == SSR + SSE? {np.isclose(SST, SSR + SSE)}")

In [ ]:
# Pie chart showing the decomposition
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
labels = [f'SSR (Explained)\n{SSR:.1f}', f'SSE (Unexplained)\n{SSE:.1f}']
sizes = [SSR, SSE]
colors = ['#2ecc71', '#e74c3c']
explode = (0.05, 0.05)
axes[0].pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
            shadow=True, startangle=90, textprops={'fontsize': 12})
axes[0].set_title(f'Variance Decomposition\nSST = {SST:.1f}', fontsize=14)

# Bar chart comparison
bars = axes[1].bar(['SST\n(Total)', 'SSR\n(Explained)', 'SSE\n(Error)'],
                   [SST, SSR, SSE], color=['#3498db', '#2ecc71', '#e74c3c'], edgecolor='black')
for bar, val in zip(bars, [SST, SSR, SSE]):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10,
                 f'{val:.1f}', ha='center', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Sum of Squares')
axes[1].set_title('SST = SSR + SSE', fontsize=14)

plt.tight_layout()
plt.show()

---
## 8. Combined Visualization: SST, SSR & SSE Side by Side

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

# --- SST ---
axes[0].scatter(X, y, color='blue', s=80, zorder=5)
axes[0].axhline(y=y_mean, color='red', linestyle='--', linewidth=2)
for i in range(len(y)):
    axes[0].vlines(X[i], y_mean, y[i], color='orange', linewidth=2, alpha=0.7)
axes[0].set_title(f'SST = {SST:.1f}\nΣ(yᵢ - ȳ)²', fontsize=13)
axes[0].set_xlabel('X')
axes[0].set_ylabel('y')

# --- SSR ---
axes[1].scatter(X, y, color='blue', s=80, zorder=5, alpha=0.3)
axes[1].plot(X, y_pred, color='green', linewidth=2)
axes[1].axhline(y=y_mean, color='red', linestyle='--', linewidth=2)
for i in range(len(y)):
    axes[1].vlines(X[i], y_mean, y_pred[i], color='green', linewidth=3, alpha=0.6)
axes[1].set_title(f'SSR = {SSR:.1f}\nΣ(ŷᵢ - ȳ)²', fontsize=13)
axes[1].set_xlabel('X')

# --- SSE ---
axes[2].scatter(X, y, color='blue', s=80, zorder=5)
axes[2].plot(X, y_pred, color='green', linewidth=2)
for i in range(len(y)):
    axes[2].vlines(X[i], y_pred[i], y[i], color='red', linewidth=2.5, alpha=0.7)
axes[2].set_title(f'SSE = {SSE:.1f}\nΣ(yᵢ - ŷᵢ)²', fontsize=13)
axes[2].set_xlabel('X')

plt.suptitle('The Fundamental Identity: SST = SSR + SSE', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 9. R² — Coefficient of Determination

### What is it?
R² tells you **what fraction of the total variance** in `y` is explained by your model.

$$R^2 = \frac{SSR}{SST} = 1 - \frac{SSE}{SST}$$

### Intuition
| R² Value | Interpretation |
|----------|----------------|
| **1.0** | Perfect model — explains 100% of variance |
| **0.9** | Excellent — explains 90% of variance |
| **0.5** | Mediocre — explains only 50% |
| **0.0** | Useless — no better than predicting the mean |
| **< 0** | Worse than the mean! (possible with bad models on test data) |

### Two Equivalent Formulas

**Formula 1:** $R^2 = \frac{SSR}{SST}$ — "How much did my model explain?"

**Formula 2:** $R^2 = 1 - \frac{SSE}{SST}$ — "How much error is left compared to total?"

### When to use R²?
- Comparing models on the **same dataset**
- Getting a **quick sense** of model quality
- Communicating results to **non-technical stakeholders**

### Limitations
- R² always increases (or stays the same) when you add more features — use **Adjusted R²** instead for multiple regression
- R² doesn't tell you if the model is **biased**
- A high R² doesn't mean the model is **correct** (could be overfitting)

In [ ]:
# Calculate R² from scratch — both formulas
R2_formula1 = SSR / SST
R2_formula2 = 1 - (SSE / SST)

# Verify with scikit-learn
R2_sklearn = r2_score(y, y_pred)

print("=" * 50)
print("R² — Coefficient of Determination")
print("=" * 50)
print(f"\nFormula 1: R² = SSR / SST = {SSR:.2f} / {SST:.2f} = {R2_formula1:.6f}")
print(f"Formula 2: R² = 1 - SSE/SST = 1 - {SSE:.2f}/{SST:.2f} = {R2_formula2:.6f}")
print(f"sklearn:   R² = {R2_sklearn:.6f}")
print(f"\nAll three match? {np.isclose(R2_formula1, R2_formula2) and np.isclose(R2_formula1, R2_sklearn)}")
print(f"\nInterpretation: The model explains {R2_formula1*100:.2f}% of the variance in salary.")

In [ ]:
# Visual: R² as a gauge / progress bar
fig, ax = plt.subplots(figsize=(12, 3))

# Full bar (SST = 100%)
ax.barh(0, 1, height=0.5, color='#e74c3c', alpha=0.3, edgecolor='black', label='Unexplained (SSE)')
# Explained portion
ax.barh(0, R2_formula1, height=0.5, color='#2ecc71', edgecolor='black', label='Explained (SSR)')

ax.set_xlim(0, 1.1)
ax.set_yticks([])
ax.set_xlabel('Proportion of Total Variance', fontsize=12)
ax.set_title(f'R² = {R2_formula1:.4f} — Model explains {R2_formula1*100:.1f}% of variance', fontsize=14)

# Add text annotations
ax.text(R2_formula1/2, 0, f'R² = {R2_formula1:.3f}', ha='center', va='center', fontsize=14, fontweight='bold', color='white')
ax.text(R2_formula1 + (1-R2_formula1)/2, 0, f'{(1-R2_formula1):.3f}', ha='center', va='center', fontsize=12, color='red')

ax.legend(loc='upper right', fontsize=11)
plt.tight_layout()
plt.show()

---
## 10. What Does R² Look Like for Good vs. Bad Models?

Let's see R² in action across different noise levels.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
noise_levels = [0, 3, 15, 50]
titles = ['Perfect Fit', 'Low Noise', 'High Noise', 'Almost Random']

np.random.seed(42)
X_demo = np.linspace(0, 10, 30).reshape(-1, 1)
y_true = 3 * X_demo.ravel() + 10

for ax, noise, title in zip(axes, noise_levels, titles):
    y_noisy = y_true + np.random.normal(0, noise, len(y_true))
    
    m = LinearRegression().fit(X_demo, y_noisy)
    y_p = m.predict(X_demo)
    r2 = r2_score(y_noisy, y_p)
    
    ax.scatter(X_demo, y_noisy, alpha=0.6, s=40)
    ax.plot(X_demo, y_p, color='red', linewidth=2)
    ax.set_title(f'{title}\nR² = {r2:.4f}', fontsize=12)
    ax.set_xlabel('X')

axes[0].set_ylabel('y')
plt.suptitle('R² Across Different Noise Levels', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 11. Adjusted R²

### The Problem with R²
R² **never decreases** when you add more features — even useless ones! This can be misleading.

### The Fix: Adjusted R²
Adjusted R² penalizes for adding features that don't improve the model.

$$\text{Adjusted } R^2 = 1 - \frac{(1 - R^2)(n - 1)}{n - p - 1}$$

Where:
- $n$ = number of samples
- $p$ = number of features

### When to use?
Always prefer Adjusted R² when comparing models with **different numbers of features**.

In [ ]:
def adjusted_r2(r2, n, p):
    """Calculate Adjusted R²."""
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

n = len(y)  # number of samples
p = 1       # number of features

adj_r2 = adjusted_r2(R2_sklearn, n, p)

print(f"R²:          {R2_sklearn:.6f}")
print(f"Adjusted R²: {adj_r2:.6f}")
print(f"\nWith n={n} samples and p={p} feature(s)")
print(f"Adjusted R² is slightly lower, penalizing for model complexity.")

# Show effect of adding random (useless) features
print("\n--- Effect of Adding Random Features ---")
print(f"{'# Features':>12} {'R²':>10} {'Adj R²':>10}")
print("-" * 35)

np.random.seed(42)
X_expanded = X.copy()
for num_feat in range(1, 8):
    if num_feat > 1:
        X_expanded = np.hstack([X_expanded, np.random.randn(n, 1)])
    m = LinearRegression().fit(X_expanded, y)
    r2_val = r2_score(y, m.predict(X_expanded))
    adj_r2_val = adjusted_r2(r2_val, n, num_feat)
    print(f"{num_feat:>12} {r2_val:>10.6f} {adj_r2_val:>10.6f}")

print("\nNotice: R² keeps increasing, but Adjusted R² drops when useless features are added!")

---
## 12. MAE — Mean Absolute Error

### What is it?
MAE is the **average of the absolute differences** between predicted and actual values.

$$MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

### Intuition
**"On average, how far off are my predictions?"**

MAE is in the **same units as the target variable**, making it very interpretable.
- If predicting salary in $1000s and MAE = 3.2, your predictions are off by **$3,200 on average**.

### MAE vs MSE (Mean Squared Error)

| Property | MAE | MSE |
|----------|-----|-----|
| **Formula** | Mean of \|errors\| | Mean of errors² |
| **Units** | Same as target | Squared units |
| **Outlier sensitivity** | Robust | Sensitive (squares amplify large errors) |
| **Differentiable?** | No (at 0) | Yes everywhere |
| **When to prefer** | Outliers in data, want interpretability | Want to penalize large errors more |

### When to use MAE?
- When you want an **easily interpretable** error metric
- When **outliers** are present and you don't want them to dominate
- When all errors should be weighted **equally** regardless of size

In [ ]:
# Calculate MAE step by step
absolute_errors = np.abs(y - y_pred)
MAE = np.mean(absolute_errors)

print(f"Step-by-step MAE calculation:")
print(f"{'y_i':>6} {'ŷ_i':>8} {'y_i - ŷ_i':>10} {'|y_i - ŷ_i|':>12}")
print("-" * 40)
for i in range(len(y)):
    print(f"{y[i]:6.1f} {y_pred[i]:8.2f} {(y[i]-y_pred[i]):10.2f} {absolute_errors[i]:12.2f}")
print("-" * 40)
print(f"Sum of |errors|: {np.sum(absolute_errors):.2f}")
print(f"n (count):       {len(y)}")
print(f"MAE = {np.sum(absolute_errors):.2f} / {len(y)} = {MAE:.4f}")

# Verify with sklearn
MAE_sklearn = mean_absolute_error(y, y_pred)
print(f"\nsklearn MAE: {MAE_sklearn:.4f}")
print(f"Match? {np.isclose(MAE, MAE_sklearn)}")
print(f"\nInterpretation: On average, predictions are off by ${MAE*1000:.0f}")

In [ ]:
# Visualize MAE
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

X_flat = X.ravel()

# Left: Scatter plot with error bars
axes[0].scatter(X_flat, y, color='blue', s=100, zorder=5, label='Actual')
axes[0].plot(X_flat, y_pred, color='green', linewidth=2, label='Predicted')
for i in range(len(y)):
    xi = float(X_flat[i])
    axes[0].annotate('', xy=(xi + 0.15, float(y[i])), xytext=(xi + 0.15, float(y_pred[i])),
                     arrowprops=dict(arrowstyle='<->', color='red', lw=1.5))
    axes[0].text(xi + 0.25, float(y[i] + y_pred[i]) / 2, f'{absolute_errors[i]:.1f}',
                fontsize=9, color='red', va='center')

axes[0].axhline(y=np.mean(y_pred) + MAE, color='orange', linestyle=':', alpha=0.5)
axes[0].axhline(y=np.mean(y_pred) - MAE, color='orange', linestyle=':', alpha=0.5)
axes[0].set_xlabel('Experience (years)')
axes[0].set_ylabel('Salary ($1000s)')
axes[0].set_title(f'Absolute Errors for Each Point\nMAE = {MAE:.2f} ($1000s)', fontsize=13)
axes[0].legend()

# Right: Bar chart of individual absolute errors
colors = ['#e74c3c' if e > MAE else '#3498db' for e in absolute_errors]
bars = axes[1].bar(range(1, 11), absolute_errors, color=colors, edgecolor='black', alpha=0.8)
axes[1].axhline(y=MAE, color='orange', linewidth=2, linestyle='--', label=f'MAE = {MAE:.2f}')
axes[1].set_xlabel('Data Point')
axes[1].set_ylabel('|Error|')
axes[1].set_title('Individual Absolute Errors\n(Red = above MAE, Blue = below MAE)', fontsize=13)
axes[1].legend(fontsize=12)

plt.tight_layout()
plt.show()

---
## 13. MAE vs MSE vs RMSE — A Comparison

Let's compare all three error metrics and see how they react to outliers.

In [ ]:
MSE = mean_squared_error(y, y_pred)
RMSE = np.sqrt(MSE)

print("Metric Comparison (Original Data):")
print("=" * 40)
print(f"MAE  = {MAE:.4f}  (in $1000s)")
print(f"MSE  = {MSE:.4f}  (in $1000s²) — hard to interpret!")
print(f"RMSE = {RMSE:.4f}  (in $1000s) — comparable to MAE")
print(f"\nRMSE >= MAE always: {RMSE >= MAE}")
print(f"If RMSE >> MAE, there are large outlier errors.")
print(f"RMSE/MAE ratio: {RMSE/MAE:.2f} (closer to 1 = errors are uniform)")

In [ ]:
# Demonstrate outlier sensitivity
print("\n--- Outlier Sensitivity Demo ---\n")

y_with_outlier = y.copy()
y_with_outlier[-1] = 150  # Make last point an extreme outlier (was 90)

m_outlier = LinearRegression().fit(X, y_with_outlier)
y_pred_outlier = m_outlier.predict(X)

mae_orig = mean_absolute_error(y, y_pred)
mse_orig = mean_squared_error(y, y_pred)
mae_outl = mean_absolute_error(y_with_outlier, y_pred_outlier)
mse_outl = mean_squared_error(y_with_outlier, y_pred_outlier)

print(f"{'Metric':<8} {'Original':>12} {'With Outlier':>14} {'% Change':>12}")
print("-" * 48)
print(f"{'MAE':<8} {mae_orig:>12.2f} {mae_outl:>14.2f} {(mae_outl-mae_orig)/mae_orig*100:>11.1f}%")
print(f"{'MSE':<8} {mse_orig:>12.2f} {mse_outl:>14.2f} {(mse_outl-mse_orig)/mse_orig*100:>11.1f}%")
print(f"{'RMSE':<8} {np.sqrt(mse_orig):>12.2f} {np.sqrt(mse_outl):>14.2f} {(np.sqrt(mse_outl)-np.sqrt(mse_orig))/np.sqrt(mse_orig)*100:>11.1f}%")
print(f"\nNotice: MSE changes much more dramatically than MAE with outliers!")

---
## 14. All Metrics on a Real Dataset (Boston-style Housing)

Let's apply everything we've learned to a realistic dataset.

In [ ]:
from sklearn.model_selection import train_test_split

# Generate a realistic housing dataset
np.random.seed(42)
n_samples = 200

sqft = np.random.uniform(800, 3500, n_samples)
bedrooms = np.random.randint(1, 6, n_samples)
age = np.random.uniform(0, 50, n_samples)

# Price = f(sqft, bedrooms, age) + noise
price = 50 + 0.12 * sqft + 15 * bedrooms - 0.8 * age + np.random.normal(0, 30, n_samples)

X_house = np.column_stack([sqft, bedrooms, age])
y_house = price

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_house, y_house, test_size=0.2, random_state=42)

# Fit model
model_house = LinearRegression()
model_house.fit(X_train, y_train)
y_train_pred = model_house.predict(X_train)
y_test_pred = model_house.predict(X_test)

print("Housing Model Coefficients:")
for name, coef in zip(['Sqft', 'Bedrooms', 'Age'], model_house.coef_):
    print(f"  {name}: {coef:.4f}")
print(f"  Intercept: {model_house.intercept_:.4f}")

In [ ]:
# Calculate ALL metrics for train and test
def calculate_all_metrics(y_true, y_pred, n_features):
    """Calculate SST, SSR, SSE, R², Adjusted R², MAE, MSE, RMSE."""
    n = len(y_true)
    y_mean = np.mean(y_true)
    
    sst = np.sum((y_true - y_mean) ** 2)
    ssr = np.sum((y_pred - y_mean) ** 2)
    sse = np.sum((y_true - y_pred) ** 2)
    
    r2 = 1 - sse / sst
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - n_features - 1)
    mae = np.mean(np.abs(y_true - y_pred))
    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    
    return {'SST': sst, 'SSR': ssr, 'SSE': sse, 'R²': r2,
            'Adj R²': adj_r2, 'MAE': mae, 'MSE': mse, 'RMSE': rmse}

train_metrics = calculate_all_metrics(y_train, y_train_pred, 3)
test_metrics = calculate_all_metrics(y_test, y_test_pred, 3)

print(f"{'Metric':<10} {'Train':>14} {'Test':>14}")
print("=" * 40)
for metric in ['SST', 'SSR', 'SSE', 'R²', 'Adj R²', 'MAE', 'MSE', 'RMSE']:
    print(f"{metric:<10} {train_metrics[metric]:>14.2f} {test_metrics[metric]:>14.2f}")

print(f"\nKey Takeaways:")
print(f"• R² (test) = {test_metrics['R²']:.3f} — model explains {test_metrics['R²']*100:.1f}% of variance on unseen data")
print(f"• MAE (test) = {test_metrics['MAE']:.2f} — avg prediction is off by ${test_metrics['MAE']*1000:.0f}")
print(f"• Train R² > Test R²? {train_metrics['R²'] > test_metrics['R²']} (expected — slight overfitting is normal)")

In [ ]:
# Final comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Actual vs Predicted
axes[0, 0].scatter(y_test, y_test_pred, alpha=0.6, color='steelblue', edgecolor='black', s=50)
min_val, max_val = min(y_test.min(), y_test_pred.min()), max(y_test.max(), y_test_pred.max())
axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0, 0].set_xlabel('Actual Price')
axes[0, 0].set_ylabel('Predicted Price')
axes[0, 0].set_title(f'Actual vs Predicted (Test)\nR² = {test_metrics["R²"]:.4f}', fontsize=13)
axes[0, 0].legend()

# 2. Residuals Distribution
test_residuals = y_test - y_test_pred
axes[0, 1].hist(test_residuals, bins=15, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 1].axvline(x=0, color='red', linewidth=2, linestyle='--')
axes[0, 1].set_xlabel('Residual (Actual - Predicted)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title(f'Residual Distribution\nMAE = {test_metrics["MAE"]:.2f}', fontsize=13)

# 3. Variance Decomposition (Pie)
labels = [f'SSR\n{test_metrics["SSR"]:.0f}', f'SSE\n{test_metrics["SSE"]:.0f}']
sizes = [test_metrics['SSR'], test_metrics['SSE']]
colors = ['#2ecc71', '#e74c3c']
axes[1, 0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
               shadow=True, startangle=90, textprops={'fontsize': 11})
axes[1, 0].set_title(f'Variance Decomposition (Test)\nSST = {test_metrics["SST"]:.0f}', fontsize=13)

# 4. Metric Comparison Bar Chart
metric_names = ['R²', 'MAE', 'RMSE']
train_vals = [train_metrics[m] for m in metric_names]
test_vals = [test_metrics[m] for m in metric_names]
x_pos = np.arange(len(metric_names))
width = 0.35
axes[1, 1].bar(x_pos - width/2, train_vals, width, label='Train', color='#3498db', edgecolor='black')
axes[1, 1].bar(x_pos + width/2, test_vals, width, label='Test', color='#e67e22', edgecolor='black')
axes[1, 1].set_xticks(x_pos)
axes[1, 1].set_xticklabels(metric_names)
axes[1, 1].set_title('Train vs Test Metrics', fontsize=13)
axes[1, 1].legend()

plt.suptitle('Comprehensive Model Evaluation Dashboard', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 15. Quick Reference & Summary

### Formulas at a Glance

| Metric | Formula | Range | Best Value |
|--------|---------|-------|------------|
| **SST** | $\sum(y_i - \bar{y})^2$ | [0, ∞) | N/A (data property) |
| **SSR** | $\sum(\hat{y}_i - \bar{y})^2$ | [0, SST] | = SST |
| **SSE** | $\sum(y_i - \hat{y}_i)^2$ | [0, SST] | 0 |
| **R²** | $1 - \frac{SSE}{SST}$ | (-∞, 1] | 1 |
| **Adj R²** | $1 - \frac{(1-R^2)(n-1)}{n-p-1}$ | (-∞, 1] | 1 |
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | [0, ∞) | 0 |
| **MSE** | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ | [0, ∞) | 0 |
| **RMSE** | $\sqrt{MSE}$ | [0, ∞) | 0 |

### Decision Guide: Which Metric to Use?

| Scenario | Best Metric | Why |
|----------|-------------|-----|
| Explaining to stakeholders | **R²** | Easy to understand as percentage |
| Comparing models with different # features | **Adjusted R²** | Penalizes unnecessary complexity |
| Need interpretable error in original units | **MAE** | Same units as target, robust to outliers |
| Want to penalize large errors more | **MSE / RMSE** | Squaring amplifies big mistakes |
| Checking if model beats the baseline | **R²** | R² < 0 means worse than just using the mean |

### Key Relationships
- **SST = SSR + SSE** (always true for OLS)
- **R² = SSR / SST = 1 - SSE / SST**
- **RMSE ≥ MAE** (always; equal when all errors are the same)
- **RMSE / MAE → 1** means errors are uniform; **RMSE / MAE → √n** means one huge outlier error

In [ ]:
print("Notebook complete!")
print("You now understand SST, SSR, SSE, R², Adjusted R², MAE, MSE, and RMSE.")
print("\nKey takeaways:")
print("  1. SST = SSR + SSE — the fundamental variance decomposition")
print("  2. R² = proportion of variance explained (0 to 1 for good models)")
print("  3. MAE = average prediction error in original units (robust to outliers)")
print("  4. Always check BOTH R² and error metrics (MAE/RMSE) for a complete picture")
print("  5. Use Adjusted R² when comparing models with different numbers of features")